## 5. Retrieval
### The retrieval step in Retrieval-Augmented Generation (RAG) is the process of fetching relevant information from an external knowledge source (like a database or document store) based on the user’s query.


### 5.1 Corrective RAG (CRAG)
#### Corrective-RAG (CRAG) is a strategy for RAG that incorporates self-reflection / self-grading on retrieved documents
- If at least one document exceeds the threshold for relevance, then it proceeds to generation2. Before generation, it performs knowledge refinement
- This partitions the document into "knowledge strips"
- It grades each strip, and filters our irrelevant ones
- If all documents fall below the relevance threshold or if the grader is unsure, then the framework seeks an additional datasource
- It will use web search to supplement retrieval

`Notebook`

https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag.ipynb

### 5.2 Self-RAG (SRAG)
#### Self-RAG is a strategy for RAG that incorporates self-reflection / self-grading on retrieved documents and generations.

`Notebook` : https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_self_rag.ipynb

`Some of the questions the SRAG asks itself in the process `
1. Should I retrieve from retriever, R:
    - Input: x (question) OR x (question), y (generation)
    - Decides when to retrieve D chunks with R
    - Output: yes, no, continue
2. Are the retrieved passages D relevant to the question x:
    - Input: (x (question), d (chunk)) for d in D
    - d provides useful information to solve x
    - Output: relevant, irrelevant
3. Are the LLM generation from each chunk in D is relevant to the chunk (hallucinations, etc):
    - Input: x (question), d (chunk), y (generation) for d in D
    - All of the verification-worthy statements in y (generation) are supported by d
    - Output: fully supported, partially supported, no support
4. The LLM generation from each chunk in D is a useful response to x (question):
    - Input: x (question), y (generation) for d in D
    - y (generation) is a useful response to x (question).
    - Output: {5, 4, 3, 2, 1}


In [2]:
import os
from dotenv import load_dotenv
load_dotenv() 
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] =  os.environ.get('LANGCHAIN_API_KEY')
os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY')

In [3]:
api_key = os.environ.get('GEMINI_API_KEY')

#### 5.2.1 LLM Retriever

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import time

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=10
)
doc_splits = text_splitter.split_documents(docs_list)

class RateLimitedEmbeddings(GoogleGenerativeAIEmbeddings):
    def embed_documents(self, texts, batch_size=5):  # smaller batch
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            while True:
                try:
                    embeddings = super().embed_documents(batch)
                    all_embeddings.extend(embeddings)
                    time.sleep(1)  # 1s delay between batches
                    break
                except Exception as e:
                    if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                        print(f"Rate limit hit, waiting 60s... (batch {i//batch_size + 1})")
                        time.sleep(60)  # wait 60s then retry
                    else:
                        raise e
        return all_embeddings

vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=RateLimitedEmbeddings(
        model="models/gemini-embedding-001",
        api_key=api_key
    )
)
retriever = vectorstore.as_retriever()

USER_AGENT environment variable not set, consider setting it to identify your requests.


Rate limit hit, waiting 60s... (batch 9)


#### 5.2.2 Retriever Grader

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field  # use pydantic v2 directly
from langchain_google_genai import ChatGoogleGenerativeAI

# Data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

# LLM with function call
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key, temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# Prompt
system = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader

question = "agent memory"
docs = retriever.invoke(question)  # fixed
doc_txt = docs[1].page_content
print(retrieval_grader.invoke({"question": question, "document": doc_txt}))

binary_score='yes'


#### 5.2.3 Generation

In [9]:
from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser

prompt = hub.pull("rlm/rag-prompt")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=api_key,temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = prompt | llm | StrOutputParser()

generation = rag_chain.invoke({"context":docs,"question":question})
print(generation)

Agent memory is a crucial component in LLM-powered autonomous agent systems, enabling them to acquire, store, retain, and retrieve information. It consists of short-term memory, which utilizes in-context learning and is limited by the model's context window. Long-term memory allows agents to retain and recall information over extended periods, often implemented using an external vector store and fast retrieval mechanisms.


#### 5.2.4 Hallucination Grader

In [12]:
# Data Model
class GradeHallucinations(BaseModel):
    """Binary Score for hallucination present in the generated answer."""
    
    binary_score:str = Field(
        description = "Answer is grounded in the facts,'yes' or 'no'"
    )
    
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=api_key,temperature=0)
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

system = """You are a grader assessing whether an LLM generated answer is based on / supported by a set of retrieved facts.\n
Give a binary score 'yes' or 'no'.'Yes' means that the answer is grounded in / supported by the set of the facts."""
hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system),
        ("human","Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader
hallucination_grader.invoke({"documents": docs, "generation": generation})

GradeHallucinations(binary_score='yes')

#### 5.2.5 Answer Grader

In [13]:
# Data Model
class GradeAnswer(BaseModel):
    """Gives Binary score to whether the generated answer addresses the question."""
    
    binary_score:str = Field(
        description = "Answer addresses the question, 'yes' or 'no'"
    )
    
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=api_key,temperature=0)
structured_llm_grader = llm.with_structured_output(GradeAnswer)

#Prompt
system = """You area grader assessing whether the generated answer addresses / resolves the question or not.\n
Give a binary score of yes or no. 'Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)
answer_grader = answer_prompt | structured_llm_grader
answer_grader.invoke({"question":question,"generation":generation})

GradeAnswer(binary_score='yes')

#### 5.2.6 Question Rewriter

In [15]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=api_key,temperature=0)
system = """You are a question re-writer that converts an input question to a better version that is optimized\n
for vector retrieval.Look at the input question and try to reason about the underlying semantic intent / meaning."""

rewrite_prompt = ChatPromptTemplate(
    [
        ("system",system),
        ("human","Here is the initial question : {question} \n Formulate a single improved question."),
    ]
)

question_rewriter = rewrite_prompt | llm | StrOutputParser()
question_rewriter.invoke({"question":question})

'What are the fundamental concepts, types, and architectural designs of memory systems for artificial intelligence agents, and how do they facilitate learning and decision-making?'

#### 5.2.7 Graph State

In [16]:
from typing import List
from typing_extensions import TypedDict

class GraphState(TypedDict):
    """Represents the state of out Graph

    Args:
        question : question
        generation : LLM generation
        documents : list of documents
    """
    question : str
    generation : str
    documents : List[str]

#### 5.2.8 Nodes of the Graph

In [17]:
# Nodes
def retrieve(state):
    """Retrieve Documents

    Args:
        state (dict): current state of the graph
    Returns:
        state (dict): New key added to the state,documents that contains retrieved documents
    """
    print("---RETRIEVE---")
    question = state["question"]
    
    documents = retriever._get_relevant_documents(question)
    return{"documents":documents,"question":question}

def generate(state):
    """Generate Answer

    Args:
        state (dict): current state of the graph
    Returns:
        state (dict): New key added to the state,generation, that contains the LLM Generation
    """
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]
    
    generation = rag_chain.invoke({"context":documents,"question":question})
    return{"documents":documents,"question":question,"generation":generation}

def grade_documents(state):
    """Determines whether the retrieved documents are relevant to the question.

    Args:
        state (dict): current state of the graph
    Returns:
        state (dict): Updates documents key with only filtered documents
    """
    print("---CHECK DOCUMENT RELEVANCE---")
    question = state["question"]
    documents = state["documents"]
    
    filtered_docs = []
    for d in documents:
        score = retrieval_grader.invoke({"question":question,"document":d.page_content})
        grade = score.binary_score
        if grade == "yes":
            print("--DOCUMENT IS RELEVANT--")
            filtered_docs.append(d)
        else:
             print("--DOCUMENT IS NOT RELEVANT--")
    return{"documents":filtered_docs,"question":question}
    
def transform_query(state):
    """Transforms the question into a more optimzed question

    Args:
        state (dict): current state of the graph
    Returns:
        state (dict): Updates question key with the updated, more relevant question
    """
    print("--TRANSFORM QUERY--")
    question = state["question"]
    documents = state["docuemnts"]
    
    better_question = question_rewriter.invoke({"question":question})
    return{"documents":documents,"question":better_question}

#### 5.2.9 Edges of the Graph

In [18]:
def decide_to_generate(state):
    """Decides whether to generate an answer, or regenerate the question.

    Args:
        state (dict) : The current state graph
    Returns:
        state (dict) : Binary decision for the next model to call
    """
    print("--ASSESS GRADED DOCUMENTS--")
    question = state["question"]
    filtered_documents = state["documents"]
    
    if not filtered_documents:
        print("--DECISION: ALL DOCUMENTS ARE IRRELEVANT TO THE QUESTION, TRANSFORM QUERY--")
        return "transform_query"
    else:
        print("--DECISION: GENERATE--")
        return "generate"
    
def grade_generation_v_documents_and_question(state):
    """
    Determines whether the generation is grounded in the document and answers question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Decision for next node to call
    """
    print("--CHECK HALLUCINATIONS--")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]
    
    score = hallucination_grader.invoke({"documents":documents,"question":question})
    grade = score.binary_score
    
    if grade == "yes":
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        # Check question-answering
        print("---GRADE GENERATION vs QUESTION---")
        score = answer_grader.invoke({"question": question, "generation": generation})
        grade = score.binary_score
        if grade == "yes":
            print("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        else:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION---")
            return "not useful"
    else:
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"

#### 5.2.10 Building the Graph